!kaggle datasets download -d olistbr/brazilian-ecommerce

In [6]:
import zipfile

with zipfile.ZipFile("brazilian-ecommerce.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [1]:
# Olist Customer Behavior Analysis
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#1. Data Loading

customers = pd.read_csv("data/raw/olist_customers_dataset.csv")
order_items = pd.read_csv("data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("data/raw/olist_orders_dataset.csv")
products = pd.read_csv("data/raw/olist_products_dataset.csv")
product_category = pd.read_csv("data/raw/product_category.csv")

In [2]:
datasets = {
    "customers": customers,
    "order_items": order_items,
    "order_reviews": order_reviews,
    "order_payments": order_payments,
    "orders": orders,
    "products": products,
    "product_category": product_category
} 

In [3]:
# 2.EDA
for name, df in datasets.items():
    print(f"{name} :{df.shape}")
    print("------------------")
    print(df.dtypes)
    print("------------------")
    print(df.isnull().sum())

customers :(99441, 5)
------------------
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
------------------
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
order_items :(112650, 7)
------------------
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object
------------------
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
order_reviews :(99224, 7)
------------------
review_id                  object
order_id                   

In [4]:
datetime_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date' ]
for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col])

In [5]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [6]:
# 3.Data Cleaning
product_translation = products.merge(product_category, on = 'product_category_name', how='left')

In [7]:
product_translation.head(2)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art


In [7]:
product_translation['product_category_name_english'].isnull().sum()

np.int64(623)

In [8]:
product_translation = products.merge(product_category, on='product_category_name', how='left')
product_translation['product_category_name_english'] = product_translation['product_category_name_english'].fillna('unknown')

In [9]:
order_items = order_items.drop(columns = ['seller_id', 'shipping_limit_date'])

In [10]:
order_items.head(1)

,order_id,order_item_id,product_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,58.9,13.29


In [11]:
datetime_cols = ['review_creation_date','review_answer_timestamp']
for col in datetime_cols:
    order_reviews[col] = pd.to_datetime(order_reviews[col])

In [12]:
order_reviews.dtypes

review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

In [13]:
master = orders.merge(
    customers, on='customer_id', how='left'
).merge(
    order_items, on='order_id', how='left'
).merge(
    order_payments, on='order_id', how='left'
).merge(
    order_reviews, on='order_id', how='left'
)

In [14]:
master.shape

(119143, 26)

In [15]:
master['order_id'].nunique()

99441

In [16]:
master.isnull().sum()

order_id                              0
customer_id                           0
order_status                          0
order_purchase_timestamp              0
order_approved_at                   177
order_delivered_carrier_date       2086
order_delivered_customer_date      3421
order_estimated_delivery_date         0
customer_unique_id                    0
customer_zip_code_prefix              0
customer_city                         0
customer_state                        0
order_item_id                       833
product_id                          833
price                               833
freight_value                       833
payment_sequential                    3
payment_type                          3
payment_installments                  3
payment_value                         3
review_id                           997
review_score                        997
review_comment_title             105154
review_comment_message            68898
review_creation_date                997


In [25]:
master.duplicated().sum()

np.int64(0)

In [19]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(r"C:\Users\Admin\Desktop\cv_project\.env")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as connection:
    print("Connected to MySQL successfully!")

Connected to MySQL successfully!
